[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/056.%20The%20Safety%20Stock%20%26%20Reorder%20Point%20%28Q%2Cr%29%20Policy%20Problem/P56-Tier-1_executed.ipynb) [![Open In SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open In Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 56. The Safety Stock and Reorder Point \((Q,r)\) Policy Problem

## Tier 1: Mathematical Formulation

### Purpose and scope

This notebook calculates a continuous-review inventory policy for CardioStabil. The policy has two main decisions: the order quantity \(Q\), placed whenever inventory position reaches the reorder point \(r\), and the reorder point \(r\), which protects against demand while an order is in transit. The notebook calculates safety stock from a target cycle service level, sets the reorder point, calculates the classical economic order quantity (EOQ), estimates annual cost components, and examines how safety stock and the reorder point respond to selected inputs.

The calculation is analytical; it is not a network-flow optimization model. EOQ balances annual ordering cost with cycle-stock holding cost. Safety stock and the reorder point are calculated separately from the service-level target. Consequently, the resulting policy is not claimed to be a joint global optimum of all possible inventory policies or of the full cost function including stockout cost.

### Model assumptions

- Daily demand is represented by a normal distribution with known mean and standard deviation. Demand over the lead time is approximated as normal.
- Daily demand and lead time are independent. The lead time has a known mean and standard deviation.
- The target service level is a **cycle service level**: the probability that demand during one replenishment lead time does not exceed the reorder point. It is not a fill rate, which measures the fraction of demand served immediately from stock.
- Holding cost is linear in average inventory, and ordering cost is a fixed cost per order.
- Stockout cost is linear in the number of units short. Expected shortage is estimated from the normal lead-time-demand model.
- The EOQ approximation uses average cycle stock \(Q/2\), assumes a steady demand rate, and does not account for quantity discounts, capacity limits, or other real-world constraints.

The model does not explicitly represent supplier capacity, backorder timing, perishability, or uncertainty in the cost parameters. These assumptions define the scope of the calculations rather than a complete operational simulation.

### Step-by-step approach

1. Represent lead-time-demand uncertainty using demand variability and lead-time variability.
2. Convert the target cycle service level into a normal quantile and calculate safety stock.
3. Set the reorder point equal to expected lead-time demand plus safety stock.
4. Calculate EOQ from annual demand, ordering cost, and annual holding cost per unit.
5. Estimate annual holding, ordering, and stockout costs for the rounded policy parameters.
6. Visualize the policy and examine the effects of demand variability, lead-time variability, and service level.

### Lead-time demand and safety stock

Let daily demand have mean \(\mu_D\) and standard deviation \(\sigma_D\), and let lead time have mean \(\mu_L\) and standard deviation \(\sigma_L\), measured in days. Under the independence and normal-approximation assumptions, lead-time demand has approximate mean and standard deviation

$$
\mu_{LT}=\mu_D\mu_L,
\qquad
\sigma_{LT}=\sqrt{\mu_L\sigma_D^2+\mu_D^2\sigma_L^2}.
$$

The first variance term reflects demand variation over an average-length lead time. The second reflects the extra demand uncertainty caused by variation in lead time. For target cycle service level \(\alpha\), let \(z=\Phi^{-1}(\alpha)\), where \(\Phi\) is the standard normal cumulative distribution function. Safety stock and the reorder point are

$$
SS=z\sigma_{LT},
\qquad
r=\mu_{LT}+SS.
$$

Here, \(SS\) is safety stock in units and \(r\) is the inventory-position threshold in units. In the code, `calculate_safety_stock` evaluates the uncertainty formula and `calculate_reorder_point` adds expected lead-time demand.

### Economic order quantity

Let \(D\) be annual demand, \(K\) the fixed cost of placing an order, and \(h\) the annual holding cost per unit. With orders of size \(Q\), annual ordering cost is \(DK/Q\), while average cycle-stock holding cost is \(hQ/2\). Minimizing their sum gives the classical EOQ:

$$
Q^*=\sqrt{\frac{2DK}{h}}.
$$

The safety-stock holding cost does not depend on \(Q\) when the reorder point is held fixed, so it does not change this classical EOQ calculation. The code implements this expression in `calculate_economic_order_quantity`.

### Expected shortage and annual cost estimate

A cycle service level gives the probability of avoiding a stockout in a cycle; by itself, it does not specify how many units are short when a stockout occurs. To estimate a unit-based stockout cost, let \(X\) denote normal lead-time demand with mean \(\mu_{LT}\) and standard deviation \(\sigma_{LT}\). For reorder point \(r\), define \(z_r=(r-\mu_{LT})/\sigma_{LT}\). Expected shortage per replenishment cycle is

$$
n(r)=\mathbb{E}[(X-r)^+]
=\sigma_{LT}\,\phi(z_r)-(r-\mu_{LT})\,[1-\Phi(z_r)],
$$

where \((x)^+=\max(x,0)\), and \(\phi\) and \(\Phi\) are the standard normal probability density and cumulative distribution functions. If the shortage penalty is \(c_s\) dollars per unit and the policy places approximately \(D/Q\) orders per year, the estimated annual stockout cost is \((D/Q)c_s n(r)\). The code uses the rounded reorder point when evaluating this cost.

The annual cost estimate combines this stockout term with the usual average-inventory and ordering terms:

$$
C_{\mathrm{annual}}
=h\left(\frac{Q}{2}+SS\right)
+K\frac{D}{Q}
+c_s\frac{D}{Q}n(r).
$$

This estimate uses average inventory \(Q/2+SS\). Its EOQ component balances ordering and cycle-stock holding costs; the separately calculated service-level policy determines the safety stock and reorder point.

### CardioStabil example data

| Input | Value | Units or interpretation |
|---|---:|---|
| Annual demand | 12,000 | vials per year |
| Daily demand coefficient of variation | 0.3 | standard deviation divided by mean |
| Mean lead time | 14 | days |
| Lead-time standard deviation | 3 | days |
| Unit cost | $450 | per vial |
| Annual holding-cost rate | 25% | of unit cost per year |
| Ordering cost | $200 | per order |
| Stockout cost | $2,000 | per unit short |
| Target cycle service level | 99.5% | probability of no stockout during lead time |

The daily demand mean is derived from annual demand using 365 days per year; its standard deviation is derived from the stated coefficient of variation. The code calculates the corresponding normal quantile rather than relying on a rounded value in the narrative.

### Interpreting the results

The safety stock and reorder point show the protection implied by the selected service-level target and uncertainty assumptions. EOQ describes the order-size balance between ordering and cycle-stock holding costs. The cost breakdown is an estimate under the assumptions above, not evidence that the calculated policy is globally optimal when all possible policies and constraints are considered. Sensitivity plots show how the calculated quantities respond to changes in selected inputs; they do not alter the CardioStabil base-case parameters.

In [ ]:
# Load the numerical, statistical, and plotting tools used throughout the analysis.
import copy
import math

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy import stats

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully.")

In [ ]:
# Store the base-case inputs together so each calculation uses a consistent parameter set.
class InventoryParameters:
    """Container for demand, lead-time, cost, and service-level inputs."""

    def __init__(self):
        # Demand is derived from the annual volume and its coefficient of variation.
        self.annual_demand = 12000
        self.daily_demand_mean = self.annual_demand / 365
        self.demand_cv = 0.3
        self.daily_demand_std = self.daily_demand_mean * self.demand_cv

        # Lead-time uncertainty contributes separately to lead-time-demand variance.
        self.lead_time_mean = 14
        self.lead_time_std = 3

        # Costs use a consistent annual basis for holding and ordering calculations.
        self.unit_cost = 450
        self.holding_cost_rate = 0.25
        self.holding_cost_per_unit = (
            self.unit_cost * self.holding_cost_rate
        )
        self.ordering_cost = 200
        self.stockout_cost = 2000

        # The cycle service target determines the normal quantile used for safety stock.
        self.service_level = 0.995
        self.z_score = stats.norm.ppf(self.service_level)


params = InventoryParameters()

print("=== CardioStabil Inventory Parameters ===")
print(f"Annual demand: {params.annual_demand:,} units")
print(
    f"Daily demand: mean = {params.daily_demand_mean:.2f}, "
    f"standard deviation = {params.daily_demand_std:.2f} units"
)
print(
    f"Lead time: mean = {params.lead_time_mean} days, "
    f"standard deviation = {params.lead_time_std} days"
)
print(f"Unit cost: ${params.unit_cost:,.2f} per unit")
print(
    f"Holding cost: ${params.holding_cost_per_unit:.2f} "
    "per unit per year"
)
print(f"Ordering cost: ${params.ordering_cost:,.2f} per order")
print(f"Stockout cost: ${params.stockout_cost:,.2f} per unit short")
print(
    f"Target cycle service level: {params.service_level:.1%} "
    f"(z = {params.z_score:.3f})"
)

In [ ]:
# Combine demand and lead-time variability to estimate lead-time-demand uncertainty.
def calculate_safety_stock(params):
    """Return safety stock and its lead-time-demand variance components.

    The calculation uses
    SS = z * sqrt(L * sigma_D**2 + mu_D**2 * sigma_L**2).
    """
    demand_variance = (
        params.lead_time_mean * params.daily_demand_std**2
    )
    lead_time_variance = (
        params.daily_demand_mean**2 * params.lead_time_std**2
    )
    combined_std = math.sqrt(demand_variance + lead_time_variance)
    safety_stock = params.z_score * combined_std

    return (
        safety_stock,
        combined_std,
        demand_variance,
        lead_time_variance,
    )


safety_stock, combined_std, demand_var, lead_time_var = (
    calculate_safety_stock(params)
)

print("=== Safety Stock Calculation ===")
print(f"Demand-variability variance component: {demand_var:.2f} units^2")
print(f"Lead-time-variability variance component: {lead_time_var:.2f} units^2")
print(f"Lead-time-demand standard deviation: {combined_std:.2f} units")
print(f"Normal quantile for target service level: {params.z_score:.3f}")
print(f"Calculated safety stock: {safety_stock:.2f} units")
print(f"Rounded safety stock: {round(safety_stock)} units")

In [ ]:
# Set the reorder threshold and calculate the classical ordering/holding-cost EOQ.
def calculate_reorder_point(params, safety_stock):
    """Return the reorder point and expected demand during mean lead time."""
    expected_lead_time_demand = (
        params.daily_demand_mean * params.lead_time_mean
    )
    reorder_point = expected_lead_time_demand + safety_stock
    return reorder_point, expected_lead_time_demand


def calculate_economic_order_quantity(params):
    """Calculate EOQ from annual demand, order cost, and annual unit holding cost."""
    eoq = math.sqrt(
        (2 * params.annual_demand * params.ordering_cost)
        / params.holding_cost_per_unit
    )
    return eoq


reorder_point, expected_lt_demand = calculate_reorder_point(
    params, safety_stock
)
eoq = calculate_economic_order_quantity(params)

print("=== Reorder Point Calculation ===")
print(f"Expected demand during mean lead time: {expected_lt_demand:.2f} units")
print(f"Safety stock: {safety_stock:.2f} units")
print(f"Reorder point: {reorder_point:.2f} units")
print(f"Rounded reorder point: {round(reorder_point)} units")

print("\n=== Economic Order Quantity Calculation ===")
print(f"Annual demand: {params.annual_demand:,} units")
print(f"Ordering cost: ${params.ordering_cost:,.2f} per order")
print(
    f"Annual holding cost per unit: "
    f"${params.holding_cost_per_unit:.2f}"
)
print(f"Calculated EOQ: {eoq:.2f} units")
print(f"Rounded EOQ: {round(eoq)} units")

In [ ]:
# Estimate expected units short per cycle from the normal lead-time-demand model.
def calculate_expected_shortage(params, reorder_point):
    """Return expected units short during one replenishment lead time.

    Lead-time demand is approximated as normal with the mean and standard
    deviation implied by the demand and lead-time inputs.
    """
    mean_lead_time_demand = (
        params.daily_demand_mean * params.lead_time_mean
    )
    _, lead_time_demand_std, _, _ = calculate_safety_stock(params)

    if lead_time_demand_std <= 0:
        return max(mean_lead_time_demand - reorder_point, 0.0)

    standardized_reorder_point = (
        reorder_point - mean_lead_time_demand
    ) / lead_time_demand_std
    expected_shortage = (
        lead_time_demand_std
        * stats.norm.pdf(standardized_reorder_point)
        - (reorder_point - mean_lead_time_demand)
        * stats.norm.sf(standardized_reorder_point)
    )
    return max(expected_shortage, 0.0)


# Combine average-inventory holding cost, ordering cost, and unit-based shortage cost.
def calculate_total_costs(params, order_quantity, reorder_point, safety_stock):
    """Estimate annual cost components for a specified (Q, r) policy."""
    if order_quantity <= 0:
        raise ValueError("Order quantity must be positive.")

    cycle_stock = order_quantity / 2
    average_inventory = cycle_stock + safety_stock
    holding_cost = average_inventory * params.holding_cost_per_unit

    orders_per_year = params.annual_demand / order_quantity
    ordering_cost = orders_per_year * params.ordering_cost

    expected_shortage_per_cycle = calculate_expected_shortage(
        params, reorder_point
    )
    stockout_cost = (
        orders_per_year
        * expected_shortage_per_cycle
        * params.stockout_cost
    )
    _, combined_std, _, _ = calculate_safety_stock(params)
    mean_lead_time_demand = (
        params.daily_demand_mean * params.lead_time_mean
    )
    achieved_cycle_service_level = stats.norm.cdf(
        (reorder_point - mean_lead_time_demand) / combined_std
    )

    total_cost = holding_cost + ordering_cost + stockout_cost

    return {
        "holding_cost": holding_cost,
        "ordering_cost": ordering_cost,
        "stockout_cost": stockout_cost,
        "total_cost": total_cost,
        "cycle_stock": cycle_stock,
        "average_inventory": average_inventory,
        "orders_per_year": orders_per_year,
        "expected_shortage_per_cycle": expected_shortage_per_cycle,
        "achieved_cycle_service_level": achieved_cycle_service_level,
    }


# Evaluate the rounded quantities that will be reported as the operating policy.
optimal_q = round(eoq)
optimal_r = round(reorder_point)
optimal_ss = round(safety_stock)

costs = calculate_total_costs(
    params, optimal_q, optimal_r, optimal_ss
)

assert optimal_q > 0
assert math.isclose(
    costs["total_cost"],
    costs["holding_cost"]
    + costs["ordering_cost"]
    + costs["stockout_cost"],
    rel_tol=1e-12,
)

print("=== Annual Cost Estimate ===")
print(f"Calculated policy: Q = {optimal_q}, r = {optimal_r}, SS = {optimal_ss}")
print(f"Cycle stock: {costs['cycle_stock']:.2f} units")
print(f"Average inventory estimate: {costs['average_inventory']:.2f} units")
print(f"Approximate orders per year: {costs['orders_per_year']:.2f}")
print(
    "Expected shortage per replenishment cycle: "
    f"{costs['expected_shortage_per_cycle']:.4f} units"
)
print(
    "Achieved cycle service level at the rounded reorder point: "
    f"{costs['achieved_cycle_service_level']:.3%}"
)
print(f"Annual holding cost: ${costs['holding_cost']:,.2f}")
print(f"Annual ordering cost: ${costs['ordering_cost']:,.2f}")
print(f"Estimated annual stockout cost: ${costs['stockout_cost']:,.2f}")
print(f"Estimated total annual cost: ${costs['total_cost']:,.2f}")

In [ ]:
# Display the policy quantities, cost estimate, uncertainty components, and service-level response.
def create_policy_visualization(
    params, optimal_q, optimal_r, optimal_ss, costs
):
    """Create a four-panel view of the calculated policy and its assumptions."""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(15, 12)
    )
    fig.suptitle(
        "Safety Stock and Reorder Point (Q, r) Policy: CardioStabil",
        fontsize=16,
        fontweight="bold",
    )

    # Inventory position includes outstanding orders, so it rises when an order is placed.
    time_periods = 60
    time = np.arange(time_periods)
    inventory_position = []
    current_position = optimal_r + optimal_q

    for day in time:
        if day > 0 and current_position <= optimal_r:
            current_position += optimal_q
        current_position -= params.daily_demand_mean
        inventory_position.append(current_position)

    ax1.plot(
        time,
        inventory_position,
        color="tab:blue",
        linewidth=2,
        label="Illustrative inventory position",
    )
    ax1.axhline(
        y=optimal_r,
        color="tab:red",
        linestyle="--",
        linewidth=2,
        label=f"Reorder point ({optimal_r} units)",
    )
    ax1.axhline(
        y=optimal_ss,
        color="tab:green",
        linestyle=":",
        linewidth=2,
        label=f"Safety-stock reference ({optimal_ss} units)",
    )
    ax1.set_xlabel("Time (days)")
    ax1.set_ylabel("Inventory position (units)")
    ax1.set_title("Illustrative Inventory-Position Cycle")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Cost proportions show the composition of the estimated annual total.
    cost_components = ["Holding", "Ordering", "Stockout"]
    cost_values = [
        costs["holding_cost"],
        costs["ordering_cost"],
        costs["stockout_cost"],
    ]
    ax2.pie(
        cost_values,
        labels=cost_components,
        autopct="%1.1f%%",
        startangle=90,
        textprops={"fontsize": 10},
    )
    ax2.set_title(
        f"Estimated Annual Cost Breakdown\n"
        f"Total: ${costs['total_cost']:,.0f}"
    )

    # These variance bars show the two contributions to lead-time-demand variance.
    _, _, demand_variance, lead_time_variance = calculate_safety_stock(params)
    component_names = ["Demand variability", "Lead-time variability"]
    component_values = [demand_variance, lead_time_variance]
    bars = ax3.bar(component_names, component_values)
    ax3.set_ylabel("Variance (units squared)")
    ax3.set_title("Lead-Time-Demand Variance Components")
    ax3.tick_params(axis="x", labelrotation=10)

    for bar, value in zip(bars, component_values):
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.0f}",
            ha="center",
            va="bottom",
            fontweight="bold",
        )

    # The curve applies the same lead-time-demand standard deviation at each service target.
    service_levels = np.linspace(0.90, 0.999, 100)
    _, combined_std, _, _ = calculate_safety_stock(params)
    safety_stocks = stats.norm.ppf(service_levels) * combined_std
    ax4.plot(
        service_levels * 100,
        safety_stocks,
        color="purple",
        linewidth=2,
        label="Calculated safety stock",
    )
    ax4.scatter(
        [params.service_level * 100],
        [safety_stock],
        color="tab:red",
        s=70,
        zorder=5,
        label=f"Base target: {params.service_level:.1%}",
    )
    ax4.set_xlabel("Target cycle service level (%)")
    ax4.set_ylabel("Safety stock (units)")
    ax4.set_title("Safety Stock versus Target Service Level")
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()


create_policy_visualization(params, optimal_q, optimal_r, optimal_ss, costs)

In [ ]:
# Vary selected inputs around the supplied base case without changing that base case.
def sensitivity_analysis(params):
    """Plot how safety stock and reorder point respond to selected inputs."""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(15, 10)
    )
    fig.suptitle(
        "Sensitivity of (Q, r) Policy Quantities",
        fontsize=14,
        fontweight="bold",
    )

    # Copy the supplied inputs so each curve is centered on this parameter set.
    demand_cvs = np.linspace(0.1, 0.5, 10)
    safety_stocks_by_cv = []
    reorder_points_by_cv = []

    for cv in demand_cvs:
        temp_params = copy.copy(params)
        temp_params.demand_cv = cv
        temp_params.daily_demand_std = (
            temp_params.daily_demand_mean * cv
        )

        temp_safety_stock, _, _, _ = calculate_safety_stock(temp_params)
        temp_reorder_point, _ = calculate_reorder_point(
            temp_params, temp_safety_stock
        )
        safety_stocks_by_cv.append(temp_safety_stock)
        reorder_points_by_cv.append(temp_reorder_point)

    ax1.plot(
        demand_cvs * 100,
        safety_stocks_by_cv,
        color="tab:blue",
        linewidth=2,
        marker="o",
    )
    ax1.set_xlabel("Daily-demand coefficient of variation (%)")
    ax1.set_ylabel("Safety stock (units)")
    ax1.set_title("Safety Stock versus Demand Variability")
    ax1.grid(True, alpha=0.3)

    ax2.plot(
        demand_cvs * 100,
        reorder_points_by_cv,
        color="tab:red",
        linewidth=2,
        marker="s",
    )
    ax2.set_xlabel("Daily-demand coefficient of variation (%)")
    ax2.set_ylabel("Reorder point (units)")
    ax2.set_title("Reorder Point versus Demand Variability")
    ax2.grid(True, alpha=0.3)

    # Changing lead-time standard deviation isolates its contribution to uncertainty.
    lead_time_stds = np.linspace(1, 7, 10)
    safety_stocks_by_lead_time = []
    reorder_points_by_lead_time = []

    for lead_time_std in lead_time_stds:
        temp_params = copy.copy(params)
        temp_params.lead_time_std = lead_time_std

        temp_safety_stock, _, _, _ = calculate_safety_stock(temp_params)
        temp_reorder_point, _ = calculate_reorder_point(
            temp_params, temp_safety_stock
        )
        safety_stocks_by_lead_time.append(temp_safety_stock)
        reorder_points_by_lead_time.append(temp_reorder_point)

    ax3.plot(
        lead_time_stds,
        safety_stocks_by_lead_time,
        color="tab:green",
        linewidth=2,
        marker="^",
    )
    ax3.set_xlabel("Lead-time standard deviation (days)")
    ax3.set_ylabel("Safety stock (units)")
    ax3.set_title("Safety Stock versus Lead-Time Variability")
    ax3.grid(True, alpha=0.3)

    ax4.plot(
        lead_time_stds,
        reorder_points_by_lead_time,
        color="purple",
        linewidth=2,
        marker="d",
    )
    ax4.set_xlabel("Lead-time standard deviation (days)")
    ax4.set_ylabel("Reorder point (units)")
    ax4.set_title("Reorder Point versus Lead-Time Variability")
    ax4.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()


sensitivity_analysis(params)

In [ ]:
# Summarize the calculated policy and connect the reported quantities to their formulas.
print("=== MATHEMATICAL FORMULATION SUMMARY ===")
print("\nCALCULATED POLICY PARAMETERS:")
print(f"   Order quantity (Q): {optimal_q:,} units")
print(f"   Reorder point (r): {optimal_r:,} units")
print(f"   Safety stock (SS): {optimal_ss:,} units")
print(f"   Target cycle service level: {params.service_level:.1%}")
print(
    "   Achieved cycle service level at rounded r: "
    f"{costs['achieved_cycle_service_level']:.3%}"
)

print("\nESTIMATED ANNUAL COST BREAKDOWN:")
print(
    f"   Holding cost: ${costs['holding_cost']:,.2f} "
    f"({costs['holding_cost'] / costs['total_cost']:.1%})"
)
print(
    f"   Ordering cost: ${costs['ordering_cost']:,.2f} "
    f"({costs['ordering_cost'] / costs['total_cost']:.1%})"
)
print(
    f"   Stockout cost: ${costs['stockout_cost']:,.2f} "
    f"({costs['stockout_cost'] / costs['total_cost']:.1%})"
)
print(f"   Total estimated annual cost: ${costs['total_cost']:,.2f}")

print("\nKEY FORMULAS:")
print(
    "   Safety stock: SS = z * sqrt(L * sigma_D^2 "
    "+ mu_D^2 * sigma_L^2)"
)
print("   Reorder point: r = mu_D * L + SS")
print("   Economic order quantity: Q* = sqrt(2 * D * K / h)")
print(
    "   Expected shortage per cycle: n(r) = sigma_LT * phi(z_r) "
    "- (r - mu_LT) * [1 - Phi(z_r)]"
)

print("\nADDITIONAL PERFORMANCE ESTIMATES:")
print(f"   Approximate orders per year: {costs['orders_per_year']:.2f}")
print(
    f"   Average inventory estimate: "
    f"{costs['average_inventory']:.2f} units"
)
print(f"   Cycle stock: {costs['cycle_stock']:.2f} units")
print(
    "   Average inventory investment: "
    f"${costs['average_inventory'] * params.unit_cost:,.2f}"
)

print(
    "\nThe results are estimates under the stated normal-approximation "
    "and EOQ assumptions."
)